
---

MongoDB Silver Transformation Decisions

Cleaning Decisions

Empty titles are converted to NULL.

Invalid duration_sec values (<= 0 or > 10800) are converted to NULL.

Duplicate values inside the tags array are removed using ARRAY_DISTINCT.

No records are deleted during the Silver transformation.


Flattening Decisions

The following nested structures are flattened:

competition.id → competition_id

competition.name → competition_name

competition.teams → competition_teams

technical.max_resolution → technical_max_resolution

technical.codec → technical_codec

stats.views → stats_views

stats.likes → stats_likes


Array Handling

The following arrays are preserved without exploding:

languages

tags

competition_teams


This keeps one row per video.

Business Validation

Potential business inconsistencies are flagged, not corrected.

Business flags include:

flag_invalid_views_likes

flag_missing_live_match

flag_invalid_content_status


These flags are intended for downstream review and Gold-layer business logic.

Silver Principles

Preserve all source records.

Perform only structural cleaning.

Avoid business corrections.

Keep the dataset analytics-ready while preserving original information.

Add silver_processed_at to track the transformation timestamp.

In [9]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/09 10:14:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "mongo"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-07-13-Jul"
TABLE_NAME = "videos"

execution_date = "2026-07-13-Jul"
full_file_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}/{TABLE_NAME}.parquet"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [3]:
dfv = spark.read.parquet(full_file_path).cache()
dfv.createOrReplaceTempView("v_table")

26/08/09 10:15:00 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [4]:
# ============================================================
# MongoDB → Silver Transform
# ============================================================

silver_df ="""

WITH base AS (

    SELECT

        -- Identifiers
        video_id,
        user_id,
        match_id,

        -- Content
        content_type,
        status,

        CASE
            WHEN title IS NULL OR TRIM(title) = '' THEN NULL
            ELSE title
        END AS title,

        description,

        CASE
            WHEN duration_sec IS NULL THEN NULL
            WHEN duration_sec BETWEEN 1 AND 10800 THEN duration_sec
            ELSE NULL
        END AS duration_sec,

        created_at,
        last_updated,

        -- Competition
        competition.id    AS competition_id,
        competition.name  AS competition_name,
        competition.teams AS competition_teams,

        -- Technical
        technical.max_resolution AS technical_max_resolution,
        technical.codec          AS technical_codec,

        -- Stats
        stats.views AS stats_views,
        stats.likes AS stats_likes,

        -- Arrays
        languages,
        ARRAY_DISTINCT(tags) AS tags

    FROM v_table

)

SELECT

    video_id,
    user_id,
    match_id,

    content_type,
    status,
    title,
    description,
    duration_sec,
    created_at,
    CAST(created_at AS DATE) AS created_date,
    last_updated,
    CAST(last_updated AS DATE) AS last_updated_date,

    competition_id,
    competition_name,
    competition_teams,

    technical_max_resolution,
    technical_codec,

    stats_views,
    stats_likes,

    languages,
    tags,

    CASE
        WHEN stats_views IS NOT NULL
         AND stats_likes > stats_views
        THEN TRUE
        ELSE FALSE
    END AS flag_invalid_views_likes,

    CASE
        WHEN content_type = 'live'
         AND match_id IS NULL
        THEN TRUE
        ELSE FALSE
    END AS flag_missing_live_match,

    CASE
        WHEN (content_type = 'live'      AND status <> 'streaming')
          OR (content_type = 'replay'    AND status <> 'archived')
          OR (content_type = 'highlight' AND status <> 'ended')
        THEN TRUE
        ELSE FALSE
    END AS flag_invalid_content_status,

    CASE
    WHEN competition_id = 'RSL'
         AND (
             array_contains(tags, 'Kings_Cup')
             OR array_contains(tags, 'Super_Cup')
         )
    THEN TRUE

    WHEN competition_id = 'Kings_Cup'
         AND (
             array_contains(tags, 'RSL')
             OR array_contains(tags, 'Super_Cup')
         )
    THEN TRUE

    WHEN competition_id = 'Super_Cup'
         AND (
             array_contains(tags, 'RSL')
             OR array_contains(tags, 'Kings_Cup')
         )
    THEN TRUE

    ELSE FALSE
    END AS flag_invalid_competition_tags,
    

    current_timestamp() AS silver_processed_at

FROM base
"""

In [5]:
video_df = spark.sql(silver_df)

In [30]:
video_df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- match_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- created_date: date (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- last_updated_date: date (nullable = true)
 |-- competition_id: string (nullable = true)
 |-- competition_name: string (nullable = true)
 |-- competition_teams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- technical_max_resolution: string (nullable = true)
 |-- technical_codec: string (nullable = true)
 |-- stats_views: integer (nullable = true)
 |-- stats_likes: integer (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (n

In [31]:
video_df.show(3,truncate=False, vertical=True)

26/08/05 23:16:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------
 video_id                      | 7a3e0a16-d4ea-4c06-a776-6a6a92579b5b                                                                                                        
 user_id                       | 12e4f74a-a9a7-467d-93d3-17298cfebac0                                                                                                        
 match_id                      | match_2070                                                                                                                                  
 content_type                  | live                                                                                                                                        
 status                        | archived                                                                                         